## HT2026-DA631E-TS056
## Mini Project 1
### Author: Nancy Truong
#### Training and validation

| Fold | Training period | Evaluation period |
|---|---|---|
| 1 | Available 2017 data | 2018 |
| 2 | Available 2017–2018 data | 2019 |
| 3 | Available 2017–2019 data | 2020 |
| Final fit | Available 2017–2020 data | Available 2021 test data |

This evaluates prediction into the future. Ordinary shuffled folds would answer a different question because observations from later periods could influence predictions for earlier periods.

I compared Ridge regression and Histogram Gradient Boosting Regression (HGB). Both models were trained to predict the continuous Target, and their predictions were then used to rank stocks within each date. I also included a 20-day momentum rule as a simple ranking baseline.

Ridge is a linear regression model with an L2 penalty on its coefficients. I used it to assess whether a regularized linear relationship between the features and future returns could provide useful predictions. I tested alpha values of 0.1, 1, 10, 100, and 1000. The best Ridge configuration in the current comparison used alpha=1000 and solver="lsqr".

HGB combines trees sequentially, with each boosting iteration improving the current prediction. It can represent nonlinear relationships and interactions between features. Histogram-based splitting reduces the computational cost of fitting trees on a large dataset. The selected configuration was hgb_s3_l2_100_iter_600. Parameters: {'learning_rate': 0.05, 'max_iter': 600, 'max_depth': 6, 'l2_regularization': 100.0, 'early_stopping': False, 'max_leaf_nodes': 31, 'min_samples_leaf': 100}

The learning rate controls each tree’s contribution. Tree depth, leaf count, and minimum leaf size constrain model complexity, while L2 regularization controls the leaf updates. These settings were selected through chronological validation. I disabled internal early stopping and compared fixed iteration counts using the expanding-window folds, since the default internal validation approach is not well suited to time-series evaluation.

I evaluated the models using both return-prediction metrics and daily ranking metrics:
- MAE: Measures average absolute error in predicted returns.

- MSE: Measures squared prediction error and gives greater weight to large errors.

- R2: Compares squared error with the mean-reference benchmark used by the metric.

- Mean daily Spearman Information Coefficient: Measures whether higher predictions correspond to higher actual returns within each date.

- JPX spread Sharpe: Measures the mean weighted long–short spread relative to its daily variability.


For each date (t), the information coefficient is:

$$
IC_t=\operatorname{SpearmanCorr}
\left(\hat y_{i,t},y_{i,t}\right).
$$

The spread Sharpe is:

$$
S=\frac{\operatorname{mean}(R_t)}
{\operatorname{std}(R_t)},
$$
,where (R_t) is the weighted spread between the actual returns of the highest-ranked and lowest-ranked 200 stocks. The reported Sharpe values are unannualized.

My primary selection criterion was the mean validation spread Sharpe across the three expanding-window folds. This matches the project’s stock-ranking objective more closely than selecting solely by squared prediction error.

The selected HGB configuration achieved the highest mean validation spread Sharpe, exceeding Ridge by 0.1561 and the reference HGB by 0.0618. It also outperformed Ridge and momentum in each validation year. This supports selecting HGB for the final test evaluation.

Performance varied across periods: the selected HGB’s lowest displayed Sharpe was approximately 0.194 in 2019, compared with 0.278 in 2020. The positive results across all three folds are encouraging, but they do not establish equally strong performance in every month or on future data.

The metrics assess different aspects of performance. A model can order stocks usefully while predicting their return magnitudes inaccurately. Spread Sharpe additionally depends on actual return magnitudes and variability, so it should be interpreted alongside daily IC, prediction errors, and the behavior of the selected portfolios.

These results describe performance on the development folds used for parameter selection. Searching several configurations can make the winning validation score optimistic. Evaluation on the held-out 2021 period is therefore needed to assess generalization. The current evaluation also uses the competition’s scoring assumptions; transaction costs and execution constraints would need to be considered before drawing conclusions about an implementable trading strategy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lightgbm

In [ ]:
#!pip install pandas numpy matplotlib missingno
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



import missingno as msno
import joblib
import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import gc
import sys
import json
from IPython.display import display
import time

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = Path("/content/drive/MyDrive/Project_SupervisedLearning_1")
os.chdir(str(PROJECT_ROOT))
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))



#from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths
from modeling import (
    build_model_pipeline
    ,chronological_split
    ,make_xy
    ,save_json
    ,select_recent_dates
    ,evaluate_predictions
    ,walk_forward_split
    ,make_fold_metric_record
)
import modeling
import tuning_visualizations as tuning_viz
from features import (
    infer_feature_columns
)
from model_diagnostics import record_fold_diagnostics

config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

Project root: /content/drive/MyDrive/Project_SupervisedLearning_1
Imported load_data successfully
Imported libraries sucessfully! yay


In [ ]:
FAST_MODE = True
MODEL_NAMES = ["ridge"]

In [ ]:

features_table = pd.read_parquet(paths.processed / "features_table.parquet")
numeric_features, categorical_features = infer_feature_columns(features_table)
splits, split_report = chronological_split(features_table, config)
train = splits['train']
valid = splits['validation']

model_train = select_recent_dates(train, max_dates=config.fast_train_dates if FAST_MODE else None)

print(f"training rows used: {len(model_train):,} of {len(train)}")
print(f"training dates used: {model_train['Date'].nunique()}")

training rows used: 730,704 of 1395521
training dates used: 378


## Baseline

In [ ]:
evaluations = {}
prediction_outputs = {}
ic_outputs = {}
spread_outputs = {}

baseline_predictions = {
    "baseline_zero": np.zeros(len(valid)),
    "baseline_momentum20": valid["Momentum20Days"].fillna(0.0).to_numpy(),
}

for name, predictions in baseline_predictions.items():
    metrics, ranked, daily_ic, daily_spread_return = evaluate_predictions(
        valid,
        predictions,
        config,
        strict_portfolio_size=True,
    )
    evaluations[name] = metrics
    prediction_outputs[name] = ranked
    ic_outputs[name] = daily_ic
    spread_outputs[name] = daily_spread_return

display(pd.DataFrame(evaluations).T)

/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/cor

,MAE,MSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio
baseline_zero,0.019717,0.000842,-0.000112,NaN,NaN,0.006302,0.003330,0.539116,0.011690
baseline_momentum20,0.095870,0.020363,-23.173361,0.004797,0.004431,0.040366,0.118357,3.557257,0.011348


## Fit candidate pipelines

- **Ridge:** a stable linear benchmark with median imputation, missingness
  indicators, scaling, and one-hot categories.
- **Histogram gradient boosting:** a nonlinear model with median imputation
  and unknown-safe ordinal category encoding.

#### Walk-forward validation

Ordinary shuffled K-fold cross validation is inappropriate because it could train on future dates and validate on past dates. Instead, expanding window validation is used here:

- Fold 1: train on 2017 and validate on 2018
- Fold 2: train on 2017-2018 and validate on 2019
- Fold 3: train on 2017-2019 and validate on 2020

The last training dates are removed in every fold because each Target depends on the following two trading dates. This is to prevent data leakage.
The 2021 test data is excluded from all model selection and is opened only in notebook 05

In [ ]:
WALK_FORWARD_FOLDS = [
    {
        "name": "validate_2018"
        ,"train_end": "2017-12-31"
        ,"valid_start": "2018-01-01"
        ,"valid_end": "2018-12-31"
    }
    ,{
        "name": "validate_2019"
        ,"train_end": "2018-12-31"
        ,"valid_start": "2019-01-01"
        ,"valid_end": "2019-12-31"
    }
    ,{
        "name": "validate_2020"
        ,"train_end": "2019-12-31"
        ,"valid_start": "2020-01-01"
        ,"valid_end": "2020-12-31"
    }
]

for fold in WALK_FORWARD_FOLDS:
    assert pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["valid_start"])
    assert pd.Timestamp(fold['valid_end']) < pd.Timestamp(config.test_start)

#Training for random forest

##Random Forest — Training and Validation (2018–2020).

Train the Random Forest model using data up to December 2017, evaluate predictions on 2018,2019 and 2020, and save predictions with stock identifiers for later analysis.

In [ ]:

import time
from modeling import make_xy, build_model_pipeline

PRED_DIR = PROJECT_ROOT / "reports" / "rf_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

RF_PARAMS = {
    "n_estimators": 50,
    "max_depth": 8,
    "min_samples_leaf": 100,
    "max_features": 0.7,
    "n_jobs": 2,
}

for fold in WALK_FORWARD_FOLDS:
    year = int(fold["name"].replace("validate_", ""))

    print(f"\n--- Random Forest: {year} ---")

    train, valid, purged_dates = walk_forward_split(
        features_table=features_table,
        train_end=fold["train_end"],
        valid_start=fold["valid_start"],
        valid_end=fold["valid_end"],
        purged_dates=config.purge_dates,
    )

    X_train, y_train = make_xy(
        train, numeric_features, categorical_features
    )
    X_valid, _ = make_xy(
        valid, numeric_features, categorical_features
    )

    model = build_model_pipeline(
        "random_forest",
        numeric_features,
        categorical_features,
        model_params=RF_PARAMS,
    )

    start = time.perf_counter()
    model.fit(X_train, y_train)
    print(f"Training time: {(time.perf_counter() - start) / 60:.1f} minutes")

    predictions = model.predict(X_valid)

    saved = valid[["Date", "SecuritiesCode", "Target"]].copy()
    saved["Prediction"] = predictions

    output_path = PRED_DIR / f"rf_validation_{year}_with_ids.parquet"
    saved.to_parquet(output_path, index=False)

    print(f"Training rows: {len(train):,}")
    print(f"Validation rows: {len(valid):,}")
    print(f"Saved {len(saved):,} predictions to {output_path}")

    del model, X_train, X_valid, y_train, predictions, saved, train, valid



--- Random Forest: 2018 ---
Training time: 7.3 minutes
Training rows: 459,695
Validation rows: 467,671
Saved 467,671 predictions to /content/drive/MyDrive/Project_SupervisedLearning_1/reports/rf_predictions/rf_validation_2018_with_ids.parquet

--- Random Forest: 2019 ---
Training time: 15.9 minutes
Training rows: 927,298
Validation rows: 468,295
Saved 468,295 predictions to /content/drive/MyDrive/Project_SupervisedLearning_1/reports/rf_predictions/rf_validation_2019_with_ids.parquet

--- Random Forest: 2020 ---
Training time: 25.1 minutes
Training rows: 1,395,521
Validation rows: 480,840
Saved 480,840 predictions to /content/drive/MyDrive/Project_SupervisedLearning_1/reports/rf_predictions/rf_validation_2020_with_ids.parquet


##Verify Saved Predictions — 2018–2020

Check that each year's saved predictions file exists, contains the expected number of rows and trading days, and has no missing predictions.

In [ ]:
for year in [2018, 2019, 2020]:
    path = PRED_DIR / f"rf_validation_{year}_with_ids.parquet"

    df = pd.read_parquet(path)

    print(
        year,
        "| Rows:", len(df),
        "| Trading days:", df["Date"].nunique(),
        "| Missing predictions:", df["Prediction"].isna().sum(),
        "| File exists:", path.exists()
    )

2018 | Rows: 467671 | Trading days: 245 | Missing predictions: 0 | File exists: True
2019 | Rows: 468295 | Trading days: 241 | Missing predictions: 0 | File exists: True
2020 | Rows: 480840 | Trading days: 243 | Missing predictions: 0 | File exists: True


#End of Random Forest


In [ ]:
print([fold["name"] for fold in WALK_FORWARD_FOLDS])
print("FAST_MODE:", FAST_MODE)

['validate_2018', 'validate_2019', 'validate_2020']
FAST_MODE: True


In [ ]:
stage1_2020 = modeling.run_walk_forward_search(
    features_table=features_table,
    folds=WALK_FORWARD_FOLDS[2:],
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    config=config,
    candidates=[modeling.make_ridge_candidates()[-1]],
    helpers=modeling,
    threads=1,
    progress_csv=paths.reports / "tuning_ridge_2020.csv",
)

display(stage1_2020.fold_results)


Training fold validate_2020


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.13/dist-packages/pandas/cor

ridge_alpha_1000: {'alpha': 1000.0}


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


fit=30.0s, train Sharpe=0.1663, valid Sharpe=0.1127, valid IC=0.0057


,model_name,fold_name,train_start,train_end,validation_start,validation_end,train_purged_dates,train_full_rows,train_after_purged_rows,validation_rows,...,Train_DailySpreadReturnStd,Train_SpreadSharpeRatio,candidate_name,is_baseline,params_json,n_iter,SharpeGap,ValidICCoverage,ValidICDates,validation_purged_dates
0,baseline_momentum20,validate_2020,2017-01-04,2019-12-30,2020-01-06,2020-12-28,"[2019-12-27, 2019-12-30]",1399453,1395521,476840,...,1.448926,-0.046074,baseline_momentum20,True,{},0,-0.057422,0.995851,240,"[2020-12-29, 2020-12-30]"
1,ridge,validate_2020,2017-01-04,2019-12-30,2020-01-06,2020-12-28,"[2019-12-27, 2019-12-30]",1399453,1395521,476840,...,1.731955,0.166262,ridge_alpha_1000,False,"{""alpha"": 1000.0}",20,0.053586,0.995851,240,"[2020-12-29, 2020-12-30]"


In [ ]:
from pathlib import Path
import pandas as pd

report_dir = paths.reports

files = [
    "tuning_stage1_folds.csv",
    "tuning_stage1_remaining.csv",
    "tuning_ridge_2020.csv",
]

results = pd.concat(
    [pd.read_csv(report_dir / file) for file in files],
    ignore_index=True,
)

display(
    results[["candidate_name", "fold_name", "SpreadSharpeRatio"]]
    .sort_values(["candidate_name", "fold_name"])
)


,candidate_name,fold_name,SpreadSharpeRatio
0,baseline_momentum20,validate_2018,-0.039625
6,baseline_momentum20,validate_2019,-0.101708
12,baseline_momentum20,validate_2020,0.011348
1,ridge_alpha_0.1,validate_2018,0.051567
7,ridge_alpha_0.1,validate_2019,0.071627
2,ridge_alpha_1,validate_2018,0.051316
8,ridge_alpha_1,validate_2019,0.071629
3,ridge_alpha_10,validate_2018,0.050960
9,ridge_alpha_10,validate_2019,0.071767
4,ridge_alpha_100,validate_2018,0.051796


In [ ]:
ridge_summary = (
    results[results["candidate_name"].str.startswith("ridge_")]
    .groupby("candidate_name")
    .agg(
        mean_sharpe=("SpreadSharpeRatio", "mean"),
        worst_sharpe=("SpreadSharpeRatio", "min"),
        std_sharpe=("SpreadSharpeRatio", "std"),
        folds=("fold_name", "nunique"),
    )
    .sort_values("mean_sharpe", ascending=False)
)

display(ridge_summary)

,mean_sharpe,worst_sharpe,std_sharpe,folds
candidate_name,,,,
ridge_alpha_1000,0.079199,0.052055,0.030803,3
ridge_alpha_100,0.061653,0.051796,0.013939,2
ridge_alpha_0.1,0.061597,0.051567,0.014185,2
ridge_alpha_1,0.061472,0.051316,0.014363,2
ridge_alpha_10,0.061364,0.050960,0.014713,2


In [ ]:
results.to_csv(
    paths.reports / "tuning_ridge_all_folds.csv",
    index=False,
)

print("Saved combined Ridge tuning results.")

Saved combined Ridge tuning results.


In [ ]:

X_train, y_train = make_xy(model_train, numeric_features, categorical_features)
X_valid, y_valid = make_xy(valid, numeric_features, categorical_features)

fitted_models = {}
training_seconds = {}
fold_metric_records = []
#fold_audit_records = []
daily_ic_record = {}
daily_spread_record = {}

ranked_2020_outputs = {}
diagnostics = {}

momentum_name = "baseline_momentum20"



for fold in WALK_FORWARD_FOLDS:
    fold_name = fold['name']
    print(f"Training fold {fold_name}")
    full_train = features_table[features_table['Date'] <= fold['train_end']]
    fold_train, fold_valid, train_purged_dates = walk_forward_split(
            features_table = features_table
            ,train_end = fold['train_end']
            ,valid_start = fold['valid_start']
            ,valid_end = fold['valid_end']
            ,purged_dates = config.purge_dates
    )

    X_fold_train, y_fold_train = make_xy(fold_train, numeric_features, categorical_features)
    X_fold_valid, y_fold_valid = make_xy(fold_valid, numeric_features, categorical_features)

    assert len(X_fold_train) > 0, f"Fold {fold_name} has no training data"
    assert len(X_fold_valid) > 0, f"Fold {fold_name} has no validation data"
    assert len(X_fold_train) == len(y_fold_train), f"Fold {fold_name} has mismatched training data and labels"
    assert len(X_fold_valid) == len(y_fold_valid), f"Fold {fold_name} has mismatched validation data and labels"

    ''' Baseline model
    Predict momentum for all stocks

    At date t:
        Momentum20days is available as it is calculated from the past 20 days of returns.
        Target is not yet available

        Past prices ─────────── Date t ───── t+1 ───── t+2
                │                    │                      │
                └─ Momentum20Days ───┘                      │
                                    └──── Target ───────────┘
    '''
    momentum_predictions = fold_valid["Momentum20Days"].fillna(0.0).to_numpy()
    momentum_metrics, momentum_ranked, momentum_daily_ic, momentum_daily_spread_return = evaluate_predictions(
            df = fold_valid
            ,predictions = momentum_predictions
            ,config = config
            ,strict_portfolio_size = True
    )

    momentum_key = (momentum_name, fold_name)

    fold_metric_records.append(
        make_fold_metric_record(
            train_full = full_train
            ,train_purged = fold_train
            ,valid = fold_valid
            ,train_purged_dates = train_purged_dates
            ,model_name = momentum_name
            ,fold_name = fold_name
            ,metrics = momentum_metrics
            ,training_seconds = 0.0
        )
    )

    daily_ic_record[momentum_key] = momentum_daily_ic
    daily_spread_record[momentum_key] = momentum_daily_spread_return

    if fold_name == 'validate_2020':
        ranked_2020_outputs[momentum_key] = momentum_ranked


    '''
    Supervised model pipelines
    '''
    for model_name in MODEL_NAMES:
        print(f"\n--------------------------Training {model_name} on fold {fold_name}...------------------------------")
        '''
        Create a completely new pipeline for every fold to avoid data leakage.
        Its median, scaling and category encodings are therefore fitted only on the training data of that fold.
        '''
        pipeline = build_model_pipeline(
            model_name = model_name
            ,numeric_features = numeric_features
            ,categorical_features = categorical_features
            ,random_seed = config.random_seed
            ,fast_mode = FAST_MODE
        )
        output_key = (model_name, fold_name)
        started = time.perf_counter()
        pipeline.fit(X_fold_train, y_fold_train)
        fit_seconds = time.perf_counter() - started
        training_seconds[output_key] = fit_seconds

        train_predictions = np.asarray(pipeline.predict(X_fold_train), dtype=float).reshape(-1)
        valid_predictions = np.asarray(pipeline.predict(X_fold_valid), dtype=float).reshape(-1)

        if len(valid_predictions) != len(y_fold_valid):
            raise ValueError(f"Fold {fold_name} model {model_name} produced {len(valid_predictions)} predictions, but validation set has {len(y_fold_valid)} samples.")

        if not np.isfinite(valid_predictions).all(): #
            raise ValueError(f"Fold {fold_name} model {model_name} produced non-finite predictions.")

        train_metrics, _,_,_ = evaluate_predictions(
            df = fold_train
            ,predictions = train_predictions
            ,config = config
            ,strict_portfolio_size = True
        )

        metrics, ranked, daily_ic, daily_spread_return = evaluate_predictions(
            df = fold_valid
            ,predictions = valid_predictions
            ,config = config
            ,strict_portfolio_size = True
        )
        print("Training metrics: ", train_metrics)
        print("\nValidation metrics: ", metrics)

        print("Generalization gap of sharpe ratio: ", train_metrics['SpreadSharpeRatio'] - metrics['SpreadSharpeRatio'])

        fold_metric_records.append(
        make_fold_metric_record(
            train_full = full_train
            ,train_purged = fold_train
            ,valid = fold_valid
            ,train_purged_dates = train_purged_dates
            ,model_name = model_name
            ,fold_name = fold_name
            ,metrics = metrics
            ,training_seconds = fit_seconds
            )
        )

        daily_ic_record[output_key] = daily_ic
        daily_spread_record[output_key] = daily_spread_return

        if fold_name == 'validate_2020':
            ranked_2020_outputs[output_key] = ranked

        print(f"\nFinished {model_name} on fold {fold_name} in {training_seconds[output_key]:.1f} seconds")

        # record_fold_diagnostics(
        #     diagnostics,
        #     model_name=model_name,
        #     fold_name=fold_name,
        #     train_frame=fold_train,
        #     validation_frame=fold_valid,
        #     train_predictions=train_predictions,
        #     validation_predictions=valid_predictions,
        #     train_metrics=train_metrics,
        #     validation_metrics=metrics,
        #     daily_ic=daily_ic,
        #     daily_spread=daily_spread_return,
        #     max_points=20000,
        # )

        '''
        Fold models are temporary. Notebool 05 will refit the selected on all development data (train + validation) and save the final model to disk.
        '''
        del pipeline
        del valid_predictions
        del ranked
        del daily_ic
        del daily_spread_return

        gc.collect()

    if fold_name == 'validate_2020':
        print(f"Finished all models on fold {fold_name}.")
        validation_2020 = fold_valid.copy()

    del X_fold_train
    del y_fold_train
    del X_fold_valid
    del y_fold_valid
    del fold_train
    del fold_valid

    gc.collect() #gabage collection to free up memory after each fold




Training fold validate_2018

--------------------------Training ridge on fold validate_2018...------------------------------
Training metrics:  MAE                        0.012008
MSE                        0.000356
R2                         0.012307
MeanDailyIC                0.013723
MedianDailyIC              0.019728
MeanDailySpreadReturn      0.437549
MedianDailySpreadReturn    0.453824
DailySpreadReturnStd       1.555280
SpreadSharpeRatio          0.281331
Name: value, dtype: float64

Validation metrics:  MAE                        0.016459
MSE                        0.000603
R2                        -0.019077
MeanDailyIC               -0.011254
MedianDailyIC             -0.009308
MeanDailySpreadReturn      0.121704
MedianDailySpreadReturn    0.217179
DailySpreadReturnStd       2.031284
SpreadSharpeRatio          0.059915
Name: value, dtype: float64
Generalization gap of sharpe ratio:  0.22141672172764437

Finished ridge on fold validate_2018 in 10.0 seconds
Training fold valid

/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]



--------------------------Training ridge on fold validate_2020...------------------------------


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Training metrics:  MAE                        0.014136
MSE                        0.000466
R2                         0.004859
MeanDailyIC                0.007816
MedianDailyIC              0.010755
MeanDailySpreadReturn      0.287358
MedianDailySpreadReturn    0.352372
DailySpreadReturnStd       1.730407
SpreadSharpeRatio          0.166064
Name: value, dtype: float64

Validation metrics:  MAE                        0.019663
MSE                        0.000835
R2                         0.005695
MeanDailyIC                0.006035
MedianDailyIC              0.009107
MeanDailySpreadReturn      0.395646
MedianDailySpreadReturn    0.420234
DailySpreadReturnStd       3.465681
SpreadSharpeRatio          0.114161
Name: value, dtype: float64
Generalization gap of sharpe ratio:  0.05190245958794354

Finished ridge on fold validate_2020 in 27.5 seconds
Finished all models on fold validate_2020.


In [ ]:
fold_results = pd.DataFrame(fold_metric_records).sort_values(by=['fold_name', 'model_name']).reset_index(drop=True)
metric_columns = [
    'MAE'
    ,'MSE'
    ,'R2'
    ,'MeanDailyIC'
    ,'MedianDailyIC'
    ,'MeanDailySpreadReturn'
    ,'MedianDailySpreadReturn'
    ,'DailySpreadReturnStd'
    ,'SpreadSharpeRatio'
]

display(fold_results[['model_name', 'fold_name', *metric_columns, 'training_seconds']].set_index(['model_name', 'fold_name']))

,,MAE,MSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio,training_seconds
model_name,fold_name,,,,,,,,,,
baseline_momentum20,validate_2018,0.076274,0.015854,-25.791748,-0.006223,-0.003405,-0.079222,0.056908,1.668201,-0.047490,0.000000
ridge,validate_2018,0.016459,0.000603,-0.019077,-0.011254,-0.009308,0.121704,0.217179,2.031284,0.059915,10.007144
baseline_momentum20,validate_2019,0.071586,0.012367,-26.540884,-0.015597,-0.015323,-0.146060,-0.097125,1.455121,-0.100377,0.000000
ridge,validate_2019,0.014312,0.000454,-0.011320,0.004569,0.011377,0.115521,0.203270,1.561730,0.073970,19.787675
baseline_momentum20,validate_2020,0.095608,0.020262,-23.138724,0.004570,0.002730,0.033080,0.076718,3.543513,0.009335,0.000000
ridge,validate_2020,0.019663,0.000835,0.005695,0.006035,0.009107,0.395646,0.420234,3.465681,0.114161,27.490000


In [ ]:
holdout_2020 = fold_results.loc[
    fold_results['fold_name'].eq('validate_2020'), ['model_name', *metric_columns, 'training_seconds']].sort_values('SpreadSharpeRatio', ascending=False)
display(holdout_2020)

,model_name,MAE,MSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio,training_seconds
5,ridge,0.019663,0.000835,0.005695,0.006035,0.009107,0.395646,0.420234,3.465681,0.114161,27.49
4,baseline_momentum20,0.095608,0.020262,-23.138724,0.004570,0.002730,0.033080,0.076718,3.543513,0.009335,0.00


## Hyperparameters Tuning

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/Project_SupervisedLearning_1")
SRC_DIR = PROJECT_ROOT / "src"

print("Project exists:", PROJECT_ROOT.exists())
print("Source folder exists:", SRC_DIR.exists())

Project exists: True
Source folder exists: True


In [ ]:
sys.path.insert(0, str(SRC_DIR))
import modeling
import inspect

print(inspect.signature(modeling.run_walk_forward_search))

(features_table: 'pd.DataFrame', folds: 'Sequence[Mapping[str, Any]]', numeric_features: 'Sequence[str]', categorical_features: 'Sequence[str]', config: 'Any', candidates: 'Sequence[Candidate]', helpers: 'Any', *, include_momentum: 'bool' = True, threads: 'int' = 4, progress_csv: 'str | Path | None' = None) -> 'TuningResults'


In [ ]:
import inspect

print(inspect.getsource(modeling.run_walk_forward_search))

def run_walk_forward_search(
    features_table: pd.DataFrame
    ,folds: Sequence[Mapping[str, Any]]
    ,numeric_features: Sequence[str]
    ,categorical_features: Sequence[str]
    ,config: Any
    ,candidates: Sequence[Candidate]
    ,helpers: Any
    ,*
    ,include_momentum: bool = True
    ,threads: int = 4
    ,progress_csv: str | Path | None = None
) -> TuningResults:
    """Use helpers=modeling; fit a NEW pipeline for each candidate and fold.

    Each fold's final config.purge_dates validation dates are excluded, so labels
    at the end of 2020 cannot use 2021 prices under the configured horizon.
    All candidate comparisons use the same validation rows. Progress CSV, if
    supplied, is rewritten after each completed candidate for this run.
    """
    candidates, folds = list(candidates), list(folds)
    if not candidates or not folds:
        raise ValueError("Provide at least one candidate and one fold.")
    if len({c.name for c in candidates}) != len(candidates):
   

In [ ]:
import sys
from pathlib import Path

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

import modeling

In [ ]:
import inspect

print(inspect.signature(modeling.run_walk_forward_search))

(features_table: 'pd.DataFrame', folds: 'Sequence[Mapping[str, Any]]', numeric_features: 'Sequence[str]', categorical_features: 'Sequence[str]', config: 'Any', candidates: 'Sequence[Candidate]', helpers: 'Any', *, include_momentum: 'bool' = True, threads: 'int' = 4, progress_csv: 'str | Path | None' = None) -> 'TuningResults'


In [ ]:
import os

print("Process ID:", os.getpid())
print("Feature table shape:", features_table.shape)
print("Feature table memory (MB):",
      round(features_table.memory_usage(deep=True).sum() / 1024**2, 1))

Process ID: 45828


NameError: name 'features_table' is not defined

In [ ]:
import warnings
from scipy.stats import ConstantInputWarning

warnings.filterwarnings("ignore", category=ConstantInputWarning)

stage1_candidates = [modeling.make_ridge_candidates()[-1]]

stage1_remaining = modeling.run_walk_forward_search(
    features_table=features_table,
    folds=WALK_FORWARD_FOLDS[2:],
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    config=config,
    candidates=stage1_candidates,
    helpers=modeling,
    threads=1,
    progress_csv=paths.reports / "tuning_stage1_remaining.csv",
)

display(stage1_remaining.summary)
display(stage1.summary[comparison_columns])


Training fold validate_2020


In [ ]:
# Combine the saved Ridge results from all three folds
fold_results = pd.read_csv(
    paths.reports / "tuning_ridge_all_folds.csv"
)

display(
    fold_results[
        ["candidate_name", "fold_name", "SpreadSharpeRatio"]
    ].sort_values(["candidate_name", "fold_name"])
)

,candidate_name,fold_name,SpreadSharpeRatio
0,baseline_momentum20,validate_2018,-0.039625
6,baseline_momentum20,validate_2019,-0.101708
12,baseline_momentum20,validate_2020,0.011348
1,ridge_alpha_0.1,validate_2018,0.051567
7,ridge_alpha_0.1,validate_2019,0.071627
2,ridge_alpha_1,validate_2018,0.051316
8,ridge_alpha_1,validate_2019,0.071629
3,ridge_alpha_10,validate_2018,0.050960
9,ridge_alpha_10,validate_2019,0.071767
4,ridge_alpha_100,validate_2018,0.051796


In [ ]:
display(
    fold_results[
        [
            "candidate_name",
            "fold_name",
            "Train_MSE",
            "MSE",
            "Train_MeanDailyIC",
            "MeanDailyIC",
            "Train_SpreadSharpeRatio",
            "SpreadSharpeRatio",
        ]
    ].sort_values(["candidate_name", "fold_name"])
)

,candidate_name,fold_name,Train_MSE,MSE,Train_MeanDailyIC,MeanDailyIC,Train_SpreadSharpeRatio,SpreadSharpeRatio
0,baseline_momentum20,validate_2018,0.028579,0.015777,-0.007633,-0.005362,0.026472,-0.039625
6,baseline_momentum20,validate_2019,0.022093,0.012409,-0.006621,-0.015702,-0.013744,-0.101708
12,baseline_momentum20,validate_2020,0.018879,0.020363,-0.009944,0.004797,-0.046074,0.011348
1,ridge_alpha_0.1,validate_2018,0.000356,0.000598,0.013735,-0.012166,0.282379,0.051567
7,ridge_alpha_0.1,validate_2019,0.000471,0.000454,0.010539,0.004316,0.171682,0.071627
2,ridge_alpha_1,validate_2018,0.000356,0.000598,0.013734,-0.012165,0.282786,0.051316
8,ridge_alpha_1,validate_2019,0.000471,0.000454,0.010539,0.004316,0.171682,0.071629
3,ridge_alpha_10,validate_2018,0.000356,0.000598,0.013729,-0.012166,0.282621,0.050960
9,ridge_alpha_10,validate_2019,0.000471,0.000454,0.010538,0.004316,0.171680,0.071767
4,ridge_alpha_100,validate_2018,0.000356,0.000598,0.013685,-0.012166,0.283441,0.051796


In [ ]:
display(
    fold_results.loc[fold_results['fold_name'] == 'validate_2020',[
                "candidate_name",
                "fold_name",
                "Train_MSE",
                "MSE",
                "Train_MeanDailyIC",
                "MeanDailyIC",
                "Train_SpreadSharpeRatio",
                "SpreadSharpeRatio",
                "ValidICCoverage",
            ]].sort_values('SpreadSharpeRatio')
)

,candidate_name,fold_name,Train_MSE,MSE,Train_MeanDailyIC,MeanDailyIC,Train_SpreadSharpeRatio,SpreadSharpeRatio,ValidICCoverage
12,baseline_momentum20,validate_2020,0.018879,0.020363,-0.009944,0.004797,-0.046074,0.011348,0.995851
13,ridge_alpha_1000,validate_2020,0.000466,0.000838,0.007817,0.005719,0.166262,0.112677,0.995851


In [ ]:


base_params = dict(stage2.candidates["hgb_s2_09"].params)

stage3_candidates = [
    modeling.Candidate(
        name=f"hgb_s3_l2_{l2}_iter_{n_iter}",
        model_name="hist_gradient_boosting",
        params={
            **base_params,
            "l2_regularization": float(l2),
            "max_iter": n_iter,
        },
    )
    for l2, n_iter in [
        (100, 600),
        (100, 800),
        (300, 400),
        (300, 600),
        (300, 800),
    ]
]

stage3 = modeling.run_walk_forward_search(
    features_table=features_table,
    folds=WALK_FORWARD_FOLDS,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    config=config,
    candidates=stage3_candidates,
    helpers=modeling,
    include_momentum=False,
    threads=4,
)

# Compare every stage, including the existing reference.
all_fold_results = pd.concat(
    [
        stage1.fold_results,
        stage2.fold_results,
        stage3.fold_results,
    ],
    ignore_index=True,
)

summary = modeling.summarize_search(
    all_fold_results,
    expected_folds=len(WALK_FORWARD_FOLDS),
)

display(summary[
    [
        "mean_sharpe",
        "worst_sharpe",
        "std_sharpe",
        "mean_ic",
        "mean_mse",
    ]
])

all_candidates = {
    **stage1.candidates,
    **stage2.candidates,
    **stage3.candidates,
}

selected = all_candidates[summary.index[0]]
print("Selected candidate:", selected.name)
print("Parameters:", selected.params)

In [ ]:
import json

with open(paths.reports / "selected_model.json", "r") as f:
    selected_model = json.load(f)

print(json.dumps(selected_model, indent=2))

{
  "selected_model": "hist_gradient_boosting",
  "fast_mode": false,
  "selection_metric": "Mean SpreadSharpeRatio across expanding-window folds (3 folds in total) with hyperparameters tuning",
  "walk_forward_folds": [
    {
      "name": "validate_2018",
      "train_end": "2017-12-31",
      "valid_start": "2018-01-01",
      "valid_end": "2018-12-31"
    },
    {
      "name": "validate_2019",
      "train_end": "2018-12-31",
      "valid_start": "2019-01-01",
      "valid_end": "2019-12-31"
    },
    {
      "name": "validate_2020",
      "train_end": "2019-12-31",
      "valid_start": "2020-01-01",
      "valid_end": "2020-12-31"
    }
  ],
  "params": {
    "learning_rate": 0.05,
    "max_iter": 600,
    "max_depth": 6,
    "l2_regularization": 100.0,
    "early_stopping": false,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 100
  },
  "mean_spread_sharpe": 0.23585929099062367,
  "worst_spread_sharpe": 0.1943087656333323,
  "std_spread_sharpe": 0.041665163800495474,
  "mea

## Compare

In [ ]:
all_candidates

In [ ]:
plots = tuning_viz.plot_tuning_stages(
    stage1=stage1,
    stage2=stage2,
    stage3=stage3,
    top_n=None,             # Show every candidate.
    diagnostic_top_n=6,     # Shortlist for training–validation plots.
    save_dir="tuning_figures",
)

print("Selected supervised model:", plots["winner_name"])

display(
    plots["summary"][
        [
            "stage",
            "mean_sharpe",
            "worst_sharpe",
            "std_sharpe",
            "mean_ic",
            "mean_mse",
        ]
    ]
)

In [ ]:
cv_summary = plots["summary"][
        [
            "stage",
            "mean_sharpe",
            "worst_sharpe",
            "std_sharpe",
            "mean_ic",
            "mean_mse",
        ]
    ]
cv_summary

In [ ]:
all_candidates

In [ ]:
selected_model = selected.model_name
selected_params = selected.params
print("Selected candidate:", selected_model, selected.name)
print("Parameters:", selected_params)
selected_row = summary.loc[selected.name]


In [ ]:


selection = {
    "selected_model": selected_model
    ,"fast_mode": FAST_MODE
    ,"selection_metric": "Mean SpreadSharpeRatio across expanding-window folds (3 folds in total) with hyperparameters tuning"
    ,"walk_forward_folds": [
        {
            "name": fold["name"]
            ,"train_end": fold["train_end"]
            ,"valid_start": fold["valid_start"]
            ,"valid_end": fold["valid_end"]
        }
        for fold in WALK_FORWARD_FOLDS
    ]
    ,'params': selected_params
    ,"mean_spread_sharpe": float(selected_row["mean_sharpe"])
    ,"worst_spread_sharpe": float(selected_row["worst_sharpe"])
    ,"std_spread_sharpe": float(selected_row["std_sharpe"])
    ,"mean_daily_spearman_ic": float(selected_row["mean_ic"])
    ,"mean_mse": float(selected_row["mean_mse"])
    ,"note": (
        "Model family was selected without examining "
        "2021 test performance."
    )
}

save_json(selection, paths.reports / "selected_model.json")